# Beam's Efficiency Math: What 23B Active out of 501B Really Buys You

Companion notebook for the Oct 6, 2026 post on Reflection AI's Beam — a 501B-parameter MoE with 23B active per token, pretrained on 23.8T tokens (figures: Reflection AI via TechCrunch, Oct 5 2026). We compute:

1. The active fraction and per-token FLOPs vs a dense equivalent.
2. A toy MoE router showing how top-k sparsity concentrates capacity.
3. A relative inference-cost table against Z.ai's GLM-5.2 (744B total / 40B active).
4. KV-cache math for Beam's 1M-token context under stated illustrative assumptions.

In [ ]:
import numpy as np

# Reported figures (Reflection AI announcement, Oct 5 2026; GLM-5.2 figures from TechCrunch same piece)
BEAM_TOTAL, BEAM_ACTIVE = 501e9, 23e9
GLM_TOTAL, GLM_ACTIVE = 744e9, 40e9
BEAM_PRETRAIN_TOKENS = 23.8e12
BEAM_CTX = 1_048_576

beam_frac = BEAM_ACTIVE / BEAM_TOTAL
glm_frac = GLM_ACTIVE / GLM_TOTAL
print(f"Beam active fraction : {beam_frac:.4f}  ({100*beam_frac:.2f}% awake)")
print(f"GLM-5.2 active frac  : {glm_frac:.4f}  ({100*glm_frac:.2f}% awake)")

# Rule of thumb: per-token forward FLOPs ~ 2 x active params (one multiply + one add per weight)
beam_flops = 2 * BEAM_ACTIVE
dense_flops = 2 * BEAM_TOTAL          # hypothetical dense model with same total params
print(f"Beam per-token FLOPs       : {beam_flops/1e9:,.0f} GFLOPs")
print(f"Dense-501B per-token FLOPs : {dense_flops/1e9:,.0f} GFLOPs")
print(f"MoE savings vs dense equiv : {dense_flops/beam_flops:.1f}x")

assert abs(beam_frac - 0.04591) < 1e-4
assert 20 < dense_flops / beam_flops < 24

## 2. Toy MoE router: top-k sparsity in action

Simulate 128 experts and route each token embedding to the top-8. The active share should be 8/128 = 6.25% — the same idea Beam applies at 23/501 ≈ 4.6%.

In [ ]:
rng = np.random.default_rng(7)
N_EXPERTS, TOP_K, N_TOKENS, D = 128, 8, 4000, 64

tokens = rng.normal(size=(N_TOKENS, D))            # token representations
router = rng.normal(size=(D, N_EXPERTS)) / np.sqrt(D)  # router projection

scores = tokens @ router                           # [tokens, experts]
topk = np.argpartition(-scores, TOP_K, axis=1)[:, :TOP_K]

active_share = TOP_K / N_EXPERTS
expert_load = np.bincount(topk.ravel(), minlength=N_EXPERTS) / N_TOKENS
print(f"active share per token : {100*active_share:.2f}%")
print(f"mean expert load       : {expert_load.mean()*100:.2f}% of tokens")
print(f"max  expert load       : {expert_load.max()*100:.2f}% of tokens")
print(f"starved experts (0%)   : {(expert_load == 0).sum()} / {N_EXPERTS}")

# Every token lights exactly TOP_K experts; load sums to TOP_K per token on average
assert abs(active_share - 0.0625) < 1e-9
assert abs(expert_load.sum() - TOP_K) < 1e-9

## 3. Relative inference cost vs GLM-5.2

Reflection claims "3–4x less inference compute" than rivals. Active params explain part of the gap (attention + FFN work scales with active params); the rest comes from architecture and the serving stack. This table separates the two.

In [ ]:
active_ratio = GLM_ACTIVE / BEAM_ACTIVE          # 1.74x explained by active params alone
claim_low, claim_high = 3.0, 4.0

print(f"active-param cost ratio (GLM / Beam) : {active_ratio:.2f}x")
print(f"Reflection's claimed total gap      : {claim_low:.0f}-{claim_high:.0f}x")
print(f"residual from arch + stack          : {claim_low/active_ratio:.2f}-{claim_high/active_ratio:.2f}x")

# Pretraining scale check: 23.8T tokens is ~2x the Beam parameter count ratio
print(f"\npretrain tokens per total param  : {BEAM_PRETRAIN_TOKENS/BEAM_TOTAL:.0f}")
print(f"pretrain tokens per active param  : {BEAM_PRETRAIN_TOKENS/BEAM_ACTIVE:.0f}")

assert 1.7 < active_ratio < 1.8
assert 45 < BEAM_PRETRAIN_TOKENS / BEAM_TOTAL < 50

## 4. KV cache for a 1M-token context

Beam advertises a 1M-token window. KV cache memory per token is ~2 bytes × layers × KV-heads × head-dim × 2 (K and V). Architecture details are not public yet, so we use stated illustrative assumptions; the formula is what matters.

In [ ]:
LAYERS, KV_HEADS, HEAD_DIM = 80, 8, 128   # illustrative assumptions (arch not yet public)
BYTES = 2                                  # FP16/BF16 per element

bytes_per_token = 2 * LAYERS * KV_HEADS * HEAD_DIM * BYTES
kv_1m = bytes_per_token * BEAM_CTX / 1e9
print(f"KV cache per token : {bytes_per_token/1e3:,.1f} KB")
print(f"KV cache @ 1M ctx  : {kv_1m:,.1f} GB (illustrative)")
print(f"weights @ FP8      : {BEAM_TOTAL/1e9:,.0f} GB (all 501B stored, even though only 23B wake)")

# Sanity: cache must be order-of-100GB, not TB, under these assumptions
assert 50 < kv_1m < 500

## Takeaway

MoE sparsity turns a 501B model into a 46-GFLOP-per-token workload — but the parking lot (FP8 weights + a ~100 GB KV cache at full context) still charges by the total. Efficiency claims beyond the active-param arithmetic have to be earned in the serving stack, and Reflection's are still self-reported.